# Capítulo 13 — Hardware simulado: LEDs, sensores I2C e motores

## Missão
O livro usa um Raspberry Pi, sensores Grove e motores de verdade. Aqui você constrói um **laboratório de hardware em software**, que reproduz o comportamento real:
- pinos GPIO e LED piscando;
- barramento I2C com endereços e registradores;
- leitura de bytes crus convertida em temperatura;
- PWM controlando motor DC;
- servo por largura de pulso;
- motor de passo pela sequência das bobinas.

Quando você tiver um Raspberry Pi, a lógica é a mesma; só muda a biblioteca que fala com os pinos.

**No fim deste caderno você terá:** o módulo `meu_rover/hardware.py`, com GPIO, I2C, PWM, motor DC, servo e motor de passo simulados, e um painel de instrumentos.

**Leitura essencial (opcional, livro em inglês):**
- Controlling an LED with Python · PDF p. 505
- Understanding I2C · PDF p. 527
- Controlling a DC Motor · PDF p. 562
- Running a Servo Motor · PDF p. 569
- Making a Stepper Motor Step · PDF p. 576

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap13_hardware_simulado_leds_sensores_i2c_e_motores.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados da missão do rover)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_rover/
os.makedirs("../meu_rover", exist_ok=True)
open("../meu_rover/__init__.py", "a").close()        # meu_rover/ é o rover que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_rover")

## Ideia em 30 segundos
- **GPIO**: pinos que ligam (1) ou desligam (0).
- **I2C**: dois fios compartilhados por vários sensores; cada um tem um **endereço** (ex.: `0x48`) e registradores com bytes.
- **PWM**: liga e desliga muito rápido; a fração do tempo ligado (*duty cycle*) controla a potência.
- **Servo**: vai a um ângulo conforme a largura do pulso (~0,5 a 2,5 ms).
- **Motor de passo**: gira um passo por vez, energizando as bobinas numa sequência.

### 13.1 **Laboratório de hardware.** Escreva `hardware.py` com:
- `Pino(numero)`: `ligar()`, `desligar()`, `estado` e um histórico `(tempo, estado)` (o tempo é um relógio simulado em ms, para não esperar de verdade);
- `BarramentoI2C`: `conectar(endereco, dispositivo)`, `escanear()` (endereços presentes) e `ler_registrador(endereco, reg, n=2)`, que devolve bytes;
- `SensorTemperaturaI2C(valores)`: o registrador `0x00` devolve a temperatura atual em 2 bytes, em complemento de dois, com resolução de 1/128 °C; a cada leitura ele avança para o próximo valor;
- `MotorDC`: `ajustar(duty)` (entre -1 e 1; o sinal é o sentido) e `rpm` (máximo 120);
- `Servo`: `ir_para(angulo)` (0–180), que devolve a largura do pulso em microssegundos (500 a 2.500);
- `MotorPasso(passos_por_volta=2048)`: `girar(graus)` devolve a lista de estados das 4 bobinas (sequência de onda completa) e atualiza o `angulo`.

_A célula grava `meu_rover/hardware.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_rover/hardware.py


### 13.2 **LED em código Morse.** Com um `Pino(17)`, pisque `SOS` (S = 3 pontos, O = 3 traços). Ponto = ligado 200 ms; traço = 600 ms; pausa de 200 ms entre sinais e de 600 ms entre letras. Mostre a duração total e desenhe o sinal no tempo com `plt.step`.

### 13.3 **Escanear o I2C.** Monte um barramento com o sensor de temperatura em `0x48` (alimentado com as temperaturas de `LEITURAS`) e um dispositivo qualquer em `0x76`. Escaneie e mostre os endereços em hexadecimal. Tente ler o endereço `0x20` e trate o `OSError`.

### 13.4 **De bytes para °C.** Leia 4 vezes o registrador `0x00` do sensor `0x48`. Para cada leitura: mostre os bytes em hexadecimal, junte num inteiro de 16 bits (`int.from_bytes(..., "big")`), trate o sinal (complemento de dois: se `>= 0x8000`, subtraia `0x10000`) e divida por 128.

### 13.5 **PWM e motor DC.** Varie o *duty cycle* de -1 a 1 em passos de 0,25 e mostre a rotação. Faça uma rampa suave de 0 a 0,8 em 5 passos (aceleração gradual evita derrapar na areia).

### 13.6 **Servo.** Mova o servo da câmera para 0°, 45°, 90°, 135° e 180° e monte a tabela ângulo → pulso (µs). O que acontece se você pedir 200°?

### 13.7 **Motor de passo.** Gire 90° e mostre quantos passos foram, os 6 primeiros estados das bobinas e o ângulo. Depois gire -45° e confira o ângulo final.

### 13.8 **Painel de instrumentos.** No lugar do aplicativo Blynk do livro, faça um painel com Matplotlib (3 gráficos lado a lado): um medidor de barra da bateria (última leitura), a temperatura ao longo do dia e a distância à frente com uma faixa vermelha abaixo de 50 cm.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
